In [26]:
import pandas as pd
import pyreadr
import os
import re
import warnings
import lightning.pytorch as pl
import pandas as pd 
import seaborn as sns 
import torch 
from torch import nn, optim
from torch.utils.data import DataLoader, random_split 
from torchvision import datasets, transforms 
import tqdm as notebook_tqdm
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.callbacks import ModelCheckpoint

import os
import numpy as np
import pandas as pd
from datetime import datetime
from sentence_transformers import SentenceTransformer, losses, models
import pytorch_lightning as pl

import torch
import torch.nn as nn
import time

from transformers import get_linear_schedule_with_warmup, AutoTokenizer, AutoModel
from sklearn.metrics import classification_report
# Define the neural network
class BERTweetClassifier(nn.Module):
    def __init__(self, bertweet_model_name="vinai/bertweet-base", num_labels=3, device=None):
        super(BERTweetClassifier, self).__init__()
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.bertweet = AutoModel.from_pretrained(bertweet_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bertweet.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bertweet(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # [CLS] token
        pooled_output = self.dropout(pooled_output)
        logits = self.fc(pooled_output)
        return logits
    

In [27]:
class MNISTLightning(pl.LightningModule):
    """
        A PyTorch Lightning module for training a neural network on the 
        MNIST data.
        Cross-entropy is used for the training loss, with 
        optimization performed with Adam.
        Also included is
    """
    def __init__(self, model, learning_rate=1e-3, weight_decay=1e-4):
        

        """
        Initializes the Lightning training module.
        Parameters
        ----------
        model : torch.nn.Module
        The neural network model to train.
        learning_rate : float, optional
        Learning rate for the Adam optimizer (default is 1e-3).
        weight_decay : float, optional
        Weight decay for the Adam optimizer (default is 1e-4).
        """
        super().__init__() # initialize super lcass pl.LightningModule
        # this registers the tuning parameters of the training model,
        # which is useful for saving models, checkpointing, etc.
        # by default all arguments to __init__ will be saved. 
        # However, we don't
        # want to save the `model` in this case, because that 
        # includes the
        # model parameters as well, which is much bigger than 
        # just its tuning
        # parameters.
        self.save_hyperparameters(ignore="model")
        # learning_rate and weight_decay now available in self.hparams
        self.model = model

    def training_step(self, batch, batch_idx):
        """
        Defines a single training step.
        Logs both the training loss and accuracy at step.
        Parameters
        ----------
        batch : tuple
        A batch of training data containing images and labels.
        batch_idx : int
        Batch index. Not actually used here, but required as an input to
        correctly implement the abstract method from the super class.
        Returns
        -------
        torch.Tensor
        Training loss (scalar).
        """
        x, y = batch
        logits = self.model(x) # run the features through the model
        # compute training loss
        loss = nn.functional.cross_entropy(logits, y) 
        # compute training accuracy (for logging)
        acc = (logits.argmax(dim=1) == y).float().mean()
        # log the training loss and accuracy
        self.log(
            name="train_loss",
            value=loss,
            on_step=False, # don't log each step
            on_epoch=True, # log at the end of each epoch
            prog_bar=True, # display in progress bar
        )
        self.log(
            name="train_acc", 
            value=acc, 
            on_step=False, 
            on_epoch=True, 
            prog_bar=True
        )
        return loss # return the loss (for model parameter calibration)


    def configure_optimizers(self):
        """
        Configures the Adam optimizer.
        Returns
        -------
        torch.optim.Optimizer
        The Adam optimizer with weight decay.
        """
        optimizer = optim.Adam(
            self.model.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay,
        )
        return optimizer

Load Data using R Data and any ids present in final_combiend_labelled.csv as ids to remove

In [28]:
import os
import pyreadr
file_path = '../../data/tweets/'
df = pd.read_csv(file_path + 'final_combined_labelled.csv')
raw_tweets = pyreadr.read_r(file_path + 'raw_tweets.Rdata')
raw_tweets = raw_tweets['raw_tweets']

In [29]:
#COnvert to int and remove all the labelled ids in our labelled dataset
raw_tweets['tweet_id'] = raw_tweets['tweet_id'].astype(int)
ids = list(df['tweet_id'])
raw_test = raw_tweets[~raw_tweets['tweet_id'].isin(ids)].sample(20)

Define our bert model

In [ ]:
model = BERTweetClassifier(device='mps')

BERTweetClassifier(
  (bertweet): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(130, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNo